# LabRack Vision QA — 01 Exploration

First working demo: run a pretrained YOLO11 model on one staged rack image and inspect the raw detections before any custom data or QA logic.

> Educational prototype only. Not validated for clinical, diagnostic, or production laboratory use. Human review is required.

## 1. Environment

In [ ]:
%pip install -q ultralytics opencv-python matplotlib

import sys
print("Python:", sys.version)

## 2. Load the pretrained model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # smallest model for the first demo
print("Loaded:", model.model_name if hasattr(model, "model_name") else "yolo11n.pt")

## 3. Run detection on one staged image

Place a staged rack photo at `../input/rack_001.jpg` before running this cell.

In [ ]:
from pathlib import Path

image_path = Path("../input/rack_001.jpg")
if not image_path.exists():
    raise FileNotFoundError(f"No image at {image_path.resolve()}. Add a staged rack photo first.")

results = model(str(image_path))
result = results[0]

## 4. Inspect classes and confidence scores

In [ ]:
if result.boxes is None or len(result.boxes) == 0:
    print("No detections. Check the image, lighting, and framing.")
else:
    for box in result.boxes:
        class_name = result.names[int(box.cls)]
        confidence = float(box.conf)
        print(f"{class_name:<15} {confidence:.2f}")

## 5. Display and save the annotated result

In [ ]:
import cv2
import matplotlib.pyplot as plt

output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)

annotated_bgr = result.plot()
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

save_path = output_dir / "rack_001_annotated.jpg"
cv2.imwrite(str(save_path), annotated_bgr)
print("Saved:", save_path)

plt.figure(figsize=(10, 8))
plt.imshow(annotated_rgb)
plt.axis("off")
plt.show()

## Next steps

The stock `yolo11n.pt` classes will not match lab objects — this cell only proves the pipeline runs end to end. Next: stage and label a small dataset, build `dataset.yaml`, fine-tune, then add the QA rules layer (counts, cap vs tube comparison, empty-slot and low-confidence flags).